# Part 3 — No-show prediction

This notebook runs the reproducible scripts for Tasks 3.1–3.5. Open it with the project root as the working directory. Logistic regression uses a training-fitted scaler; model and threshold selection use training-only cross-validation before the test set is evaluated.

## Task 3.1 — Feature preparation and baseline

In [1]:
"""Task 3.1: prepare features, make the fixed split, and report the baseline."""

from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split


PROJECT_DIR = Path.cwd().resolve()
DATA_FILE = PROJECT_DIR / "appointments_clean.csv"


def main():
    # Keep the CSV's existing row order; the fixed split depends on it.
    df = pd.read_csv(DATA_FILE)

    feature_columns = [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent",
        "booking_channel",
        "visit_type",
    ]
    X = df[feature_columns].copy()
    X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})
    X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])
    y = df["no_show"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    test_no_shows = int(y_test.sum())
    baseline_accuracy = (y_test == 0).mean()  # Always predict Attended (no_show=0).

    print("Feature columns ({}):".format(X.shape[1]), list(X.columns))
    print("Training rows:", len(X_train))
    print("Test rows:", len(X_test))
    print("No-shows in test set:", test_no_shows)
    print("Overall no-show rate:", round(y.mean() * 100, 1), "%")
    print("Baseline accuracy (always predict Attended):", round(baseline_accuracy, 4))


main()


Feature columns (11): ['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']
Training rows: 792
Test rows: 198
No-shows in test set: 49
Overall no-show rate: 22.0 %
Baseline accuracy (always predict Attended): 0.7525


## Task 3.2 — Logistic regression

In [2]:
"""Task 3.2: fit scaled logistic regression and report test-set metrics."""

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


PROJECT_DIR = Path.cwd().resolve()
DATA_FILE = PROJECT_DIR / "appointments_clean.csv"


def main():
    # Match Task 3.1's feature order, encoding, and fixed split exactly.
    df = pd.read_csv(DATA_FILE)
    feature_columns = [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent",
        "booking_channel",
        "visit_type",
    ]
    X = df[feature_columns].copy()
    X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})
    X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])
    y = df["no_show"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_s, y_train)

    y_pred = model.predict(X_test_s)
    y_prob = model.predict_proba(X_test_s)[:, 1]

    print("Test-set metrics:")
    print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
    print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
    print("Precision:", round(precision_score(y_test, y_pred), 3))
    print("Recall:", round(recall_score(y_test, y_pred), 3))
    print("F1:", round(f1_score(y_test, y_pred), 3))
    print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 3))

    print("\nCoefficients (scaled features):")
    for feature, coefficient in zip(X.columns, model.coef_[0]):
        print(f"{feature}: {coefficient:+.3f}")

    reminder_index = X.columns.get_loc("reminder_sent")
    reminder_coefficient = model.coef_[0][reminder_index]
    reminder_scale = scaler.scale_[reminder_index]
    odds_ratio_per_sd = np.exp(reminder_coefficient)
    odds_ratio_no_to_yes = np.exp(reminder_coefficient / reminder_scale)

    print("\nReminder effect:")
    print(f"Odds ratio per 1 SD: {odds_ratio_per_sd:.3f}")
    print(f"Odds ratio for No -> Yes: {odds_ratio_no_to_yes:.3f}")
    print(
        "These differ because the first odds ratio is for a one-standard-deviation "
        f"change (SD = {reminder_scale:.3f}), while No -> Yes is a full one-unit change."
    )
    print(
        "Association: in this model, holding the other features constant, a reminder "
        f"is associated with {100 * (1 - odds_ratio_no_to_yes):.1f}% lower odds of a "
        "no-show; because the data are observational, this does not show that reminders cause fewer no-shows."
    )


main()


Test-set metrics:
Accuracy: 0.788
Confusion matrix:
 [[139  10]
 [ 32  17]]
Precision: 0.63
Recall: 0.347
F1: 0.447
ROC-AUC: 0.789

Coefficients (scaled features):
age: -0.254
distance_km: +0.253
fee_inr: +0.453
lead_days: +0.614
prior_no_shows: +0.650
reminder_sent: -0.581
booking_channel_App: -0.047
booking_channel_Phone: +0.216
booking_channel_Walk-In: -0.186
visit_type_Consultation: +0.016
visit_type_Follow-up: -0.016

Reminder effect:
Odds ratio per 1 SD: 0.560
Odds ratio for No -> Yes: 0.309
These differ because the first odds ratio is for a one-standard-deviation change (SD = 0.495), while No -> Yes is a full one-unit change.
Association: in this model, holding the other features constant, a reminder is associated with 69.1% lower odds of a no-show; because the data are observational, this does not show that reminders cause fewer no-shows.


## Task 3.3 — Decision trees and overfitting

In [3]:
"""Task 3.3: compare decision-tree depth and check for overfitting."""

from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier


PROJECT_DIR = Path.cwd().resolve()
DATA_FILE = PROJECT_DIR / "appointments_clean.csv"


def main():
    # Match Task 3.1's feature order, encoding, and fixed split exactly.
    df = pd.read_csv(DATA_FILE)
    feature_columns = [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent",
        "booking_channel",
        "visit_type",
    ]
    X = df[feature_columns].copy()
    X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})
    X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])
    y = df["no_show"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    trees = [
        ("Full-depth tree", DecisionTreeClassifier(random_state=42)),
        ("Depth-3 tree", DecisionTreeClassifier(max_depth=3, random_state=42)),
    ]

    for name, model in trees:
        model.fit(X_train, y_train)
        train_accuracy = accuracy_score(y_train, model.predict(X_train))
        test_accuracy = accuracy_score(y_test, model.predict(X_test))
        print(f"{name} (actual depth: {model.get_depth()}):")
        print(f"  Training accuracy: {train_accuracy:.3f}")
        print(f"  Test accuracy:     {test_accuracy:.3f}")

    print(
        "The full-depth tree's much higher training than test accuracy shows "
        "overfitting: it memorizes training patterns that do not generalize well."
    )


main()


Full-depth tree (actual depth: 15):
  Training accuracy: 1.000
  Test accuracy:     0.672
Depth-3 tree (actual depth: 3):
  Training accuracy: 0.822
  Test accuracy:     0.778
The full-depth tree's much higher training than test accuracy shows overfitting: it memorizes training patterns that do not generalize well.


## Task 3.4 — Random forest and model comparison

In [4]:
"""Task 3.4: evaluate a random forest and compare the three classifiers."""

from pathlib import Path

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier


PROJECT_DIR = Path.cwd().resolve()
DATA_FILE = PROJECT_DIR / "appointments_clean.csv"


def main():
    # Match Task 3.1's feature order, encoding, and fixed split exactly.
    df = pd.read_csv(DATA_FILE)
    feature_columns = [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent",
        "booking_channel",
        "visit_type",
    ]
    X = df[feature_columns].copy()
    X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})
    X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])
    y = df["no_show"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Scale for logistic regression, fitting the scaler on training data only.
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    logistic = LogisticRegression(max_iter=1000)
    logistic.fit(X_train_s, y_train)

    depth_3_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
    depth_3_tree.fit(X_train, y_train)

    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)

    rf_train_accuracy = accuracy_score(y_train, rf.predict(X_train))
    rf_predictions = rf.predict(X_test)
    rf_probabilities = rf.predict_proba(X_test)[:, 1]

    print("Random forest:")
    print(f"Training accuracy: {rf_train_accuracy:.3f}")
    print(f"Test accuracy: {accuracy_score(y_test, rf_predictions):.3f}")
    print(f"Test recall: {recall_score(y_test, rf_predictions):.3f}")
    print(f"Test ROC-AUC: {roc_auc_score(y_test, rf_probabilities):.3f}")

    top_importances = sorted(
        zip(X.columns, rf.feature_importances_), key=lambda item: item[1], reverse=True
    )[:3]
    print("\nTop three random-forest feature importances:")
    for feature, importance in top_importances:
        print(f"{feature}: {importance:.3f}")

    model_results = []
    for name, model, test_features in [
        ("Logistic regression", logistic, X_test_s),
        ("Depth-3 tree", depth_3_tree, X_test),
        ("Random forest", rf, X_test),
    ]:
        predictions = model.predict(test_features)
        probabilities = model.predict_proba(test_features)[:, 1]
        model_results.append(
            {
                "Model": name,
                "Accuracy": accuracy_score(y_test, predictions),
                "Recall": recall_score(y_test, predictions),
                "ROC-AUC": roc_auc_score(y_test, probabilities),
            }
        )

    comparison = pd.DataFrame(model_results)
    print("\nTest-set comparison:")
    print(comparison.to_string(index=False, formatters={
        "Accuracy": "{:.3f}".format,
        "Recall": "{:.3f}".format,
        "ROC-AUC": "{:.3f}".format,
    }))

    best_by_auc = comparison.loc[comparison["ROC-AUC"].idxmax(), "Model"]
    print(
        f"\nBy test ROC-AUC, {best_by_auc} looks strongest at ranking no-show risk; "
        "the formal model choice in Task 3.5 should use training data only."
    )


main()


Random forest:
Training accuracy: 1.000
Test accuracy: 0.758
Test recall: 0.347
Test ROC-AUC: 0.700

Top three random-forest feature importances:
lead_days: 0.221
age: 0.197
distance_km: 0.194



Test-set comparison:
              Model Accuracy Recall ROC-AUC
Logistic regression    0.788  0.347   0.789
       Depth-3 tree    0.778  0.224   0.700
      Random forest    0.758  0.347   0.700

By test ROC-AUC, Logistic regression looks strongest at ranking no-show risk; the formal model choice in Task 3.5 should use training data only.


## Task 3.5 — Cross-validation, threshold selection, and final test

In [5]:
"""Task 3.5: select a model and threshold using training-only validation."""

from pathlib import Path

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import (
    GridSearchCV,
    cross_val_predict,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier


PROJECT_DIR = Path.cwd().resolve()
DATA_FILE = PROJECT_DIR / "appointments_clean.csv"
RISK_SCORES_FILE = PROJECT_DIR / "test_risk_scores.csv"


def main():
    # Match Task 3.1's feature order, encoding, and fixed split exactly.
    df = pd.read_csv(DATA_FILE)
    feature_columns = [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent",
        "booking_channel",
        "visit_type",
    ]
    X = df[feature_columns].copy()
    X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})
    X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])
    y = df["no_show"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Tune the forest using only training folds.
    forest_search = GridSearchCV(
        RandomForestClassifier(random_state=42),
        {"n_estimators": [100, 200], "max_depth": [3, 5, 8]},
        cv=5,
        scoring="roc_auc",
    )
    forest_search.fit(X_train, y_train)
    print("Tuned random forest:")
    print("Best parameters:", forest_search.best_params_)
    print(f"Best 5-fold CV ROC-AUC: {forest_search.best_score_:.3f}")

    lr_pipe = Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=1000)),
        ]
    )
    depth_3_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
    random_forest = RandomForestClassifier(n_estimators=100, random_state=42)

    candidates = [
        ("Logistic regression", lr_pipe),
        ("Depth-3 tree", depth_3_tree),
        ("Random forest", random_forest),
    ]
    cv_scores = {}
    print("\nTraining-set 5-fold CV ROC-AUC:")
    for name, model in candidates:
        score = cross_val_score(
            model, X_train, y_train, cv=5, scoring="roc_auc"
        ).mean()
        cv_scores[name] = score
        print(f"{name}: {score:.3f}")

    cv_scores["Tuned random forest"] = forest_search.best_score_
    print(f"Tuned random forest: {forest_search.best_score_:.3f}")

    models = {
        "Logistic regression": lr_pipe,
        "Depth-3 tree": depth_3_tree,
        "Random forest": random_forest,
        "Tuned random forest": forest_search.best_estimator_,
    }
    winner_name = max(cv_scores, key=cv_scores.get)
    winner = models[winner_name]
    print(
        f"\nChosen model: {winner_name} "
        f"(highest training-set CV ROC-AUC: {cv_scores[winner_name]:.3f})."
    )

    # OOF probabilities keep each training row out of its own fold's fit.
    oof_probabilities = cross_val_predict(
        winner, X_train, y_train, cv=5, method="predict_proba"
    )[:, 1]
    threshold_results = []
    print("\nOut-of-fold training threshold comparison:")
    for threshold in (0.5, 0.35):
        flags = (oof_probabilities >= threshold).astype(int)
        precision = precision_score(y_train, flags, zero_division=0)
        recall = recall_score(y_train, flags, zero_division=0)
        f1 = f1_score(y_train, flags, zero_division=0)
        threshold_results.append(
            {"threshold": threshold, "precision": precision, "recall": recall, "f1": f1}
        )
        print(
            f"Threshold {threshold:.2f}: precision={precision:.3f}, "
            f"recall={recall:.3f}, F1={f1:.3f}, flagged={int(flags.sum())}"
        )

    chosen_threshold = max(threshold_results, key=lambda result: result["f1"])[
        "threshold"
    ]
    print(
        f"Chosen threshold: {chosen_threshold:.2f} (higher out-of-fold F1)."
    )
    print(
        "Recall matters because a missed no-show warning can leave a slot empty, "
        "while an extra follow-up call costs about a minute."
    )

    # Fit the selected model on all training rows; evaluate the test set once.
    winner.fit(X_train, y_train)
    test_probabilities = winner.predict_proba(X_test)[:, 1]
    test_flags = (test_probabilities >= chosen_threshold).astype(int)
    print("\nOne-time test-set evaluation:")
    print(f"Threshold: {chosen_threshold:.2f}")
    print(
        f"Precision: {precision_score(y_test, test_flags, zero_division=0):.3f}"
    )
    print(f"Recall: {recall_score(y_test, test_flags, zero_division=0):.3f}")
    print(f"Number flagged: {int(test_flags.sum())}")
    print(f"ROC-AUC: {roc_auc_score(y_test, test_probabilities):.3f}")

    risk_scores = pd.DataFrame(
        {
            "appointment_id": df.loc[X_test.index, "appointment_id"].to_numpy(),
            "no_show": y_test.to_numpy(),
            "risk_prob": test_probabilities,
            "flag": test_flags,
        }
    )
    risk_scores.to_csv(RISK_SCORES_FILE, index=False)
    print(f"Saved {len(risk_scores)} test rows to {RISK_SCORES_FILE.name}.")

    print(
        "In practice, a clinic would set the threshold based on how many follow-up "
        "calls the front desk can make in an evening and the cost of an empty slot."
    )


main()


Tuned random forest:
Best parameters: {'max_depth': 3, 'n_estimators': 200}
Best 5-fold CV ROC-AUC: 0.767

Training-set 5-fold CV ROC-AUC:
Logistic regression: 0.774
Depth-3 tree: 0.678


Random forest: 0.745
Tuned random forest: 0.767

Chosen model: Logistic regression (highest training-set CV ROC-AUC: 0.774).

Out-of-fold training threshold comparison:
Threshold 0.50: precision=0.575, recall=0.249, F1=0.347, flagged=73
Threshold 0.35: precision=0.513, recall=0.467, F1=0.489, flagged=154
Chosen threshold: 0.35 (higher out-of-fold F1).
Recall matters because a missed no-show warning can leave a slot empty, while an extra follow-up call costs about a minute.

One-time test-set evaluation:
Threshold: 0.35
Precision: 0.571
Recall: 0.571
Number flagged: 49
ROC-AUC: 0.789
Saved 198 test rows to test_risk_scores.csv.
In practice, a clinic would set the threshold based on how many follow-up calls the front desk can make in an evening and the cost of an empty slot.
